# Phase 2 — executable preprocessing implementation

Ruganji Prince · 04-652 Artificial Intelligence System Design

This notebook executes the approved raw-to-model-ready preprocessing with Scikit-Learn Pipeline/ColumnTransformer and all eight assignment diagnostics. Classifiers are not fitted and predictions are not regenerated. Final model/sensor/threshold decisions and the conclusion remain frozen: **No evaluated configuration satisfied all predeclared engineering criteria.**

The evidence-review notebook remains a separate artifact. Statistical diagnostics added here support assignment compliance, not retrospective model selection.

## Imports and reproducibility

The frozen Phase 2 environment is used. Root detection supports execution from the repository root or the notebook directory. Source positions remain metadata; no timestamps are constructed.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/raw/pv_fault_dataset.csv').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.phase2_compliance.study import PreprocessingStudy, CANDIDATES
from src.phase2_compliance.common import TRAIN, VALIDATION, TEST, SENSORS, SEED, load
from src.phase2_stage2.contracts import CONFIGS
from src.phase2_compliance import plots
np.random.seed(SEED)
study = PreprocessingStudy()
print('Seed:', SEED, '; classifier fitting is disabled by design.')


Seed: 42 ; classifier fitting is disabled by design.


## Raw data, schema and quality audit

The immutable CSV contains 1,373,798 source positions and seven columns. Six continuous predictors accompany nominal f_nv: 0 normal, 1 short circuit, 2 resistive degradation, 3 open circuit, 4 natural building shade. Zero conventional NaNs does not identify MCAR/MAR/MNAR. Paired irradiance/temperature zeros are retained, not silently declared missing.

In [2]:
study.load_data()
print(study.raw.head(3).to_string(index=False))


vdc1    float64
vdc2    float64
idc1    float64
idc2    float64
irr     float64
pvt     float64
f_nv      uint8
Raw shape: (1373798, 7) ; conventional missing cells: 0 ; paired zeros retained: 62
  vdc1   vdc2   idc1   idc2    irr    pvt  f_nv
0.7143 0.5550 0.0608 0.0073 1.3729 2.3816     0
0.6944 0.5427 0.0615 0.0064 1.3604 2.3816     0
0.7110 0.5583 0.0646 0.0067 1.5118 2.3883     0


## Frozen ordered partitions — constructed before learning statistics

TRAIN [0,1006129), VALIDATION [1056990,1173967), TEST [1227337,1373798). Buffers [1006129,1056990) and [1173967,1227337) exclude 104,231 positions. The 16 signal-defined blocks are not verified days/sessions. Validation block 13 selected settings and block 14 selected alert thresholds; those decisions are not repeated here. No random row split is introduced.

In [3]:
study.split_data()
assert [len(study.partitions[k]) for k in ['TRAIN','VALIDATION','TEST']] == [1006129,116977,146461]


                      interval        N                                                      index_digest
TRAIN             [0, 1006129]  1006129  86010f1f91f0653fd59a65ec55e3a3e3a7e905b6d292e7207099d8c4b58075f9
VALIDATION  [1056990, 1173967]   116977  8a7727a6679eb9bb52e6e40c689e4f4b054b9efe8f4cea32e4229af1b4be2328
TEST        [1227337, 1373798]   146461  e5dabd8b8c3ee5220c9ef21da1dbdfdbdd9aad99dfcc4c4353e698eebc753866


## Actual approved preprocessing — fit on TRAIN only

The shared Stage 2 preprocessing factory supplies the original RetainedQuality → median/physical-features/scaling and availability-indicator ColumnTransformer. No second preprocessing policy is introduced. LR F6 standardizes six numerical channels; RF E4 retains native electrical scale; LightGBM E4 adds the five approved electrical derivatives. Negative Celsius values and paired zeros remain valid observations; slightly negative irradiance follows the existing near-zero convention. Statistical outliers are retained.

Only preprocessing is fitted. VALIDATION and TEST receive transforms with the same TRAIN-fitted medians/scaler. Eligibility masks retain the original positional short-unavailability/abstention policy. Classifier training, probability prediction and threshold selection are absent.

In [4]:
study.model_ready()
assert all('model' not in pre.named_steps for pre in study.preprocessors.values())
print('Raw vs reference cleaning:', study.stats['reference_cleaning'])


LR__F6 {'TRAIN': {'shape': [1006129, 12], 'eligible_rows': 1006129, 'target_rows': 1006129, 'fitted_again': False}, 'VALIDATION': {'shape': [116977, 12], 'eligible_rows': 116977, 'target_rows': 116977, 'fitted_again': False}, 'TEST': {'shape': [146461, 12], 'eligible_rows': 146461, 'target_rows': 146461, 'fitted_again': False}}
RF__E4 {'TRAIN': {'shape': [1006129, 8], 'eligible_rows': 1006129, 'target_rows': 1006129, 'fitted_again': False}, 'VALIDATION': {'shape': [116977, 8], 'eligible_rows': 116977, 'target_rows': 116977, 'fitted_again': False}, 'TEST': {'shape': [146461, 8], 'eligible_rows': 146461, 'target_rows': 146461, 'fitted_again': False}}
LightGBM__E4 {'TRAIN': {'shape': [1006129, 13], 'eligible_rows': 1006129, 'target_rows': 1006129, 'fitted_again': False}, 'VALIDATION': {'shape': [116977, 13], 'eligible_rows': 116977, 'target_rows': 116977, 'fitted_again': False}, 'TEST': {'shape': [146461, 13], 'eligible_rows': 146461, 'target_rows': 146461, 'fitted_again': False}}
Raw vs 

## Diagnostic 1 — missingness pattern

A seed-42 sample of 5,000 TRAIN rows displays all seven variables. The full-export audit confirms zero conventional missing cells; zeros are not fabricated as missing.

In [5]:
plots.missingness(study)


Diagnostic 1 phase2_implementation/preprocessing_diagnostics/fig1_1_missingness_heatmap.png


![TRAINING ONLY missingness display; no conventional missing observations are present.](../preprocessing_diagnostics/fig1_1_missingness_heatmap.png)

TRAINING ONLY missingness display; no conventional missing observations are present.

## Diagnostic 2 — target distribution

The complete TRAIN population is shown without rebalancing. Rare hardware states motivate training-only class weights and class-specific metrics. SMOTE was not selected because synthesizing or duplicating dependent neighboring readings does not create independent fault episodes.

In [6]:
plots.target_distribution(study)
weights = load('phase2_implementation/stage1/evidence/class_weights.json')
assert all(weights['weights'][k] == 1006129/(5*n) for k,n in weights['training_counts'].items())
print('Unchanged training-derived class weights:', weights['weights'])


Diagnostic 2 phase2_implementation/preprocessing_diagnostics/fig1_2_target_distribution.png
Unchanged training-derived class weights: {'0': 0.23819200880676605, '1': 164.13197389885806, '2': 35.3337664618086, '3': 167.1310631229236, '4': 1.3134929079171536}


![TRAINING ONLY nominal class counts; class 4 is environmental shade rather than hardware failure.](../preprocessing_diagnostics/fig1_2_target_distribution.png)

TRAINING ONLY nominal class counts; class 4 is environmental shade rather than hardware failure.

## Diagnostic 3 — raw versus reference-cleaned outlier boxplots

Full TRAIN box statistics use 1.5×IQR whiskers and show fliers. Reference cleaning is the existing physical-quality policy, not winsorization. Nearly identical panels are expected because no statistical capping or row removal is selected; unusual values can represent real fault behavior.

In [7]:
plots.outliers(study)
assert study.stats['reference_cleaning']['removed_rows'] == 0
assert study.stats['reference_cleaning']['winsorized_cells'] == 0


Diagnostic 3 phase2_implementation/preprocessing_diagnostics/fig2_1_outlier_boxplots.png


![TRAINING ONLY raw/reference-cleaned comparison; unusual observations and all rows are retained.](../preprocessing_diagnostics/fig2_1_outlier_boxplots.png)

TRAINING ONLY raw/reference-cleaned comparison; unusual observations and all rows are retained.

## Diagnostics 4–6 — predictor relationships and nominal-target association

Pearson correlation includes the six TRAIN predictors only; integer-coded f_nv is excluded. All five |r|≥0.85 pairs are matched to the Phase 1 screen. Scatterplots display a uniform seed-42 sample of 20,000 TRAIN positions with full-TRAIN descriptive OLS lines.

Mutual information uses a separate seed-42 proportional stratified TRAIN sample of 50,000 observations, continuous-feature k=3 estimation and the TRAIN-fitted scaler. Target codes remain nominal. The sample preserves approximate training prevalence, rather than rebalancing any model population. These are descriptive associations under dependent/tied readings, not significance or causal claims. No model decision is changed by the ranking.

In [8]:
study.associations()
plots.correlation(study)
plots.collinear_pairs(study)
plots.feature_target(study)
assert list(study.correlation.columns) == SENSORS and 'f_nv' not in study.correlation


Training |r| >= .85 pairs: [('vdc1', 'vdc2', 0.9891027602285769), ('idc1', 'idc2', 0.952832711587032), ('idc1', 'irr', 0.9794723761665458), ('idc1', 'pvt', 0.8622889138042289), ('idc2', 'irr', 0.9559821219702014)]
Training-sample mutual information (nats): {'vdc1': 0.1786151235282165, 'vdc2': 0.22061103822354466, 'idc1': 0.276227212137653, 'idc2': 0.27698098738977794, 'irr': 0.300591903247168, 'pvt': 0.11237539740848446}
Diagnostic 4 phase2_implementation/preprocessing_diagnostics/fig3_1_correlation_matrix.png
Diagnostic 5 phase2_implementation/preprocessing_diagnostics/fig3_2_collinear_scatterplots.png
Diagnostic 6 phase2_implementation/preprocessing_diagnostics/fig3_3_target_importance.png


![TRAINING ONLY predictor Pearson matrix; flagged correlations do not automatically remove physical sensors.](../preprocessing_diagnostics/fig3_1_correlation_matrix.png)

TRAINING ONLY predictor Pearson matrix; flagged correlations do not automatically remove physical sensors.

![TRAINING ONLY five collinear pairs; 20,000 displayed positions, full-TRAIN trendlines.](../preprocessing_diagnostics/fig3_2_collinear_scatterplots.png)

TRAINING ONLY five collinear pairs; 20,000 displayed positions, full-TRAIN trendlines.

![TRAINING ONLY mutual-information association with nominal f_nv; 50,000 proportionally sampled observations.](../preprocessing_diagnostics/fig3_3_target_importance.png)

TRAINING ONLY mutual-information association with nominal f_nv; 50,000 proportionally sampled observations.

## Diagnostic 7 — algorithm-specific scaling alternatives

The 2×2 grid compares raw irradiance, StandardScaler, RobustScaler and MinMaxScaler. All learned parameters use full TRAIN. Standardization aligns LR penalty scales; it does not make a skewed distribution Gaussian. Robust/MinMax are diagnostic alternatives, not replacements for frozen pipelines. The optional log1p sensitivity was not selected and is not introduced.

In [9]:
study.scaling_and_pca()
plots.transformation_grid(study)


Exploratory PCA: {'population': 'TRAIN', 'N': 1006129, 'input_columns': ['vdc1', 'vdc2', 'idc1', 'idc2', 'irr', 'pvt'], 'scaled_with': 'Existing LR StandardScaler fitted on TRAIN only', 'explained_variance_ratio': [0.8253007587514459, 0.12637590927419656, 0.03575663896863875, 0.008328722852125044, 0.002756421992577408, 0.0014815481610162677], 'cumulative_explained_variance': [0.8253007587514459, 0.9516766680256425, 0.9874333069942813, 0.9957620298464064, 0.9985184518389838, 1.0], 'components_for_90pct': 2, 'components_for_95pct': 2, 'deployment_role': 'Exploratory diagnostic only; no physical sensor reduction and no change to frozen models.'}
Diagnostic 7 phase2_implementation/preprocessing_diagnostics/fig4_1_transformation_grid.png


![TRAINING ONLY representation/scaling comparison; native scale remains selected for the tree models.](../preprocessing_diagnostics/fig4_1_transformation_grid.png)

TRAINING ONLY representation/scaling comparison; native scale remains selected for the tree models.

## Diagnostic 8 — exploratory PCA

Full-SVD PCA is fitted on the six TRAIN predictors after TRAIN-fitted StandardScaler. Explained variance and 90%/95% cumulative thresholds are descriptive. PCA is mathematical dimensionality reduction, not physical sensor reduction; all six inputs remain required to compute these components. PCA is not inserted into the frozen models, and TEST contributes no fit statistic or diagnostic choice.

In [10]:
plots.pca(study)
assert study.pca.n_samples_ == 1006129
assert int(study.standard.n_samples_seen_) == 1006129


Diagnostic 8 phase2_implementation/preprocessing_diagnostics/fig5_1_pca_scree_plot.png


![TRAINING ONLY PCA scree/cumulative variance; physical sensor ablation remains a separate frozen experiment.](../preprocessing_diagnostics/fig5_1_pca_scree_plot.png)

TRAINING ONLY PCA scree/cumulative variance; physical sensor ablation remains a separate frozen experiment.

## Physical feature engineering and retained sensors

Approved dependencies are P1=vdc1×idc1; P2=vdc2×idc2; P_total=P1+P2; V_diff=|vdc1−vdc2|; I_diff=|idc1−idc2|. Products are RMS-based proxies, not independently measured power sensors. A derivative exists only when every parent raw channel is retained. Missingness indicators are mathematical inputs, not sensors.

In [11]:
pre = study.preprocessors['LightGBM__E4']
feature_names = pre.get_feature_names_out().tolist()
positions = [100000,250000,500000]
preview = study.matrices['LightGBM__E4']['TRAIN'][positions,:9]
print('Illustrative TRAIN source positions (metadata only):', positions)
print(pd.DataFrame(preview, columns=feature_names[:9]).to_string(index=False))
print(pd.DataFrame([{'configuration':k,'physical_channels':len(v),'channels':', '.join(v)} for k,v in CONFIGS.items()]).to_string(index=False))


Illustrative TRAIN source positions (metadata only): [100000, 250000, 500000]
 numeric__vdc1  numeric__vdc2  numeric__idc1  numeric__idc2  numeric__P1  numeric__P2  numeric__P_total  numeric__V_diff  numeric__I_diff
        0.4180         0.3387         0.0615         0.0066     0.025707     0.002235          0.027942           0.0793           0.0549
        1.1613         0.9811         0.0654         0.0078     0.075949     0.007653          0.083602           0.1802           0.0576
        1.5853         1.2922         0.0579         0.0113     0.091789     0.014602          0.106391           0.2931           0.0466
configuration  physical_channels                         channels
           F6                  6 vdc1, vdc2, idc1, idc2, irr, pvt
           E5                  5      vdc1, vdc2, idc1, idc2, irr
           E4                  4           vdc1, vdc2, idc1, idc2
           C3                  3                  idc1, idc2, irr
           C2                  2        

## Final model-ready matrices and leakage checks

Arrays remain in memory; no processed dataset or model binary is committed. Partition row counts and selected 12/8/13 mathematical-input counts are verified below. Targets remain separate nominal arrays. Validation/test matrices are transformed only; no estimator, scaler, median or PCA is fitted on them.

In [12]:
shape_rows=[]
for key, result in study.stats['model_ready'].items():
    for partition, outcome in result['partitions'].items():
        shape_rows.append({'candidate':key,'partition':partition,'rows':outcome['shape'][0],'mathematical_inputs':outcome['shape'][1],'eligible_rows':outcome['eligible_rows']})
print(pd.DataFrame(shape_rows).to_string(index=False))
assert all(r['population']=='TRAIN' and r['N']==1006129 for r in study.stats['fit_ledger'])
print('Training-only fit ledger:')
print(pd.DataFrame(study.stats['fit_ledger']).drop(columns=['index_digest']).to_string(index=False))


   candidate  partition    rows  mathematical_inputs  eligible_rows
      LR__F6      TRAIN 1006129                   12        1006129
      LR__F6 VALIDATION  116977                   12         116977
      LR__F6       TEST  146461                   12         146461
      RF__E4      TRAIN 1006129                    8        1006129
      RF__E4 VALIDATION  116977                    8         116977
      RF__E4       TEST  146461                    8         146461
LightGBM__E4      TRAIN 1006129                   13        1006129
LightGBM__E4 VALIDATION  116977                   13         116977
LightGBM__E4       TEST  146461                   13         146461
Training-only fit ledger:
                                component population       N                       raw_channels used_in_frozen_classifiers
      LR__F6 median/scaler/feature schema      TRAIN 1006129 [vdc1, vdc2, idc1, idc2, irr, pvt]                        NaN
      RF__E4 median/scaler/feature schema      T

## Implemented-decision summary and evidence

The new MI/PCA/scaling plots complete assignment diagnostics after model decisions were frozen. No diagnostic score selects a model, threshold or sensor set. The [strategy-versus-reality table](../compliance/Strategy_vs_Reality.md) distinguishes original experiment decisions from these later compliance diagnostics. Prior legacy test exposure remains disclosed; the split is not pristine or a verified calendar chronology.

The final conclusion remains: **No evaluated configuration satisfied all predeclared engineering criteria.**

In [13]:
assert len(study.figures)==8
study.save_evidence()
print('Executed eight diagnostics, preprocessing only; zero classifier fits or predictions.')


Saved compact preprocessing evidence; 0 classifier fits/predictions.
Executed eight diagnostics, preprocessing only; zero classifier fits or predictions.
